# Stage 1: Generate the fintech A/B test data

Run each cell top to bottom. Output: `data/users.csv` and `data/experiment_assignments.csv`.

In [15]:
import numpy as np
import pandas as pd
from pathlib import Path

### Cell 1: setup

In [16]:
rng = np.random.default_rng(seed=42)
N_USERS = 10_000
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

### Cell 2: users table

In [17]:
user_ids = [f"U{i:05d}" for i in range(1, N_USERS + 1)]

start, end = pd.Timestamp("2026-01-01"), pd.Timestamp("2026-06-30")
n_days = (end - start).days + 1
signup_dates = start + pd.to_timedelta(rng.integers(0, n_days, N_USERS), unit="D")

users = pd.DataFrame({
    "user_id": user_ids,
    "signup_date": signup_dates.date,
    "acquisition_channel": rng.choice(
        ["organic", "paid_social", "referral", "search_ads", "partner"],
        size=N_USERS, p=[0.30, 0.25, 0.15, 0.20, 0.10]),
    "device": rng.choice(["ios", "android", "web"], size=N_USERS, p=[0.45, 0.40, 0.15]),
    "age_band": rng.choice(["18-24", "25-34", "35-44", "45-54", "55+"],
                           size=N_USERS, p=[0.22, 0.34, 0.22, 0.14, 0.08]),
    "state": rng.choice(["IL", "NY", "CA", "TX", "FL", "OTHER"],
                        size=N_USERS, p=[0.18, 0.15, 0.17, 0.12, 0.10, 0.28]),
})

### Cell 3: experiment assignments

In [18]:
assignments = pd.DataFrame({
    "user_id": user_ids,
    "experiment_id": "EXP_001_onboarding_v2",
    "variant": rng.choice(["control", "treatment"], size=N_USERS),
    "assigned_at": users["signup_date"],
})

### Cell 4: validate + save

In [19]:
assert users["user_id"].is_unique
assert len(users) == len(assignments) == N_USERS
assert users.isna().sum().sum() == 0

users.to_csv(DATA_DIR / "users.csv", index=False)
assignments.to_csv(DATA_DIR / "experiment_assignments.csv", index=False)

### Check your work

In [20]:
users.head()

,user_id,signup_date,acquisition_channel,device,age_band,state
0,U00001,2026-01-17,organic,ios,55+,IL
1,U00002,2026-05-21,referral,ios,55+,IL
2,U00003,2026-04-29,search_ads,ios,45-54,NY
3,U00004,2026-03-21,organic,android,25-34,OTHER
4,U00005,2026-03-20,organic,android,25-34,CA


In [21]:
assignments['variant'].value_counts()

variant
control      5003
treatment    4997
Name: count, dtype: int64